### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 업스테이지의 콘텐츠 자산으로, 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.

# 2강 실습 — MCP Server 직접 만들기
**Production-Grade Agent Engineering · 2강 · Colab 실습 (Upstage Solar)**

> 목표: 우리 도메인 도구(뉴스 검색)를 **표준 MCP 프로토콜(JSON-RPC 2.0)**의 `tools/list`·`tools/call`로 노출하고,
> **Solar 에이전트**가 그 도구를 스스로 골라 호출하게 만듭니다. 마지막에 **권한 스코프(읽기/쓰기 분리)**로 안전하게 제작합니다.
>
> 산출물: ① 표준 MCP 서버 클래스 ② 프로토콜 raw 호출 확인 ③ Solar 에이전트의 도구 호출 루프 ④ write 도구 승인 게이트



## 0. 셋업 — Solar 클라이언트
Colab에서 `UPSTAGE_API_KEY`는 좌측 🔑(보안 비밀)에 등록해 두고 불러옵니다. **키를 코드에 직접 쓰지 마세요.**

In [1]:
# ── 셋업 — Solar 클라이언트 ──
# Colab에서 UPSTAGE_API_KEY는 좌측 🔑(보안 비밀)에 등록해 두고 불러옵니다. 키를 코드에 직접 쓰지 마세요.
!pip -q install openai

In [2]:
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get('UPSTAGE_API_KEY')

client = OpenAI(api_key=api_key, base_url="https://api.upstage.ai/v1")
MODEL = "solar-pro3"
print("Solar 준비 완료:", MODEL)

Solar 준비 완료: solar-pro3


## 1. 도메인 도구 정의
에이전트가 쓸 평범한 파이썬 함수들입니다. 아직 MCP와 무관한 '그냥 함수'예요.

In [3]:
# ── 도메인 도구 정의 ──
# 에이전트가 쓸 평범한 파이썬 함수들입니다. 아직 MCP와 무관한 '그냥 함수'예요.
NEWS_DB = {
    "AI 반도체": ["A사 차세대 AI 가속기 공개 — 추론비용 40%↓", "B사 온디바이스 AI 칩 양산"],
    "에이전트":  ["MCP 표준 채택 가속", "멀티에이전트 협업 프로토콜 A2A 공개"],
}
def search_news(keyword: str) -> list[str]:
    """키워드로 뉴스 헤드라인을 검색한다."""
    return NEWS_DB.get(keyword, [])

def publish_note(title: str, body: str) -> dict:
    """사내 게시판에 글을 발행한다(쓰기 작업)."""
    return {"status": "published", "title": title, "url": f"/board/{abs(hash(title))%9999}"}

print(search_news("AI 반도체"))

['A사 차세대 AI 가속기 공개 — 추론비용 40%↓', 'B사 온디바이스 AI 칩 양산']


## 2. 표준 MCP 서버 만들기 (JSON-RPC 2.0)
MCP는 **JSON-RPC 2.0** 위에서 동작합니다. 핵심 메서드는 `initialize`, **`tools/list`**(도구 목록·스키마), **`tools/call`**(도구 실행).
표준 오류코드도 그대로 씁니다: `-32601`(메서드/도구 없음), `-32602`(잘못된 파라미터).

In [4]:
# ── 표준 MCP 서버 만들기 (JSON-RPC 2.0) ──
# MCP는 JSON-RPC 2.0 위에서 동작합니다. 핵심 메서드는 initialize, tools/list(도구 목록·스키마), …
import json

class MCPServer:
    """최소 구현 MCP 서버 — 표준 JSON-RPC 2.0 메시지를 처리한다."""
    def __init__(self, name="news-mcp"):
        self.name = name
        self.tools = {}   # name -> {fn, description, inputSchema, write}

    def register(self, name, description, input_schema, *, write=False):
        def deco(fn):
            self.tools[name] = {"fn": fn, "description": description,
                                "inputSchema": input_schema, "write": write}
            return fn
        return deco

    def handle(self, request: dict) -> dict:
        rid, method, params = request.get("id"), request.get("method"), request.get("params", {})
        ok  = lambda result: {"jsonrpc": "2.0", "id": rid, "result": result}
        err = lambda code, msg: {"jsonrpc": "2.0", "id": rid, "error": {"code": code, "message": msg}}

        if method == "initialize":
            return ok({"protocolVersion": "2024-11-05",
                       "serverInfo": {"name": self.name, "version": "1.0"}})
        if method == "tools/list":
            return ok({"tools": [
                {"name": n, "description": t["description"], "inputSchema": t["inputSchema"]}
                for n, t in self.tools.items()]})
        if method == "tools/call":
            name = params.get("name"); args = params.get("arguments", {})
            if name not in self.tools:
                return err(-32601, f"Unknown tool: {name}")
            try:
                result = self.tools[name]["fn"](**args)
            except TypeError as e:
                return err(-32602, f"Invalid params: {e}")
            return ok({"content": [{"type": "text", "text": json.dumps(result, ensure_ascii=False)}]})
        return err(-32601, f"Method not found: {method}")

server = MCPServer()
print("MCP 서버 생성:", server.name)

MCP 서버 생성: news-mcp


도구를 MCP 서버에 등록합니다. `inputSchema`(JSON Schema)가 곧 **에이전트에게 주는 사용설명서**입니다. `publish_note`는 `write=True`로 표시해 둡니다(뒤에서 권한 분리에 사용).

In [5]:
@server.register("search_news", "키워드로 뉴스 헤드라인을 검색한다",
    {"type": "object", "properties": {"keyword": {"type": "string", "description": "검색 키워드"}},
     "required": ["keyword"]})
def _search(keyword): return search_news(keyword)

@server.register("publish_note", "사내 게시판에 글을 발행한다(쓰기)",
    {"type": "object", "properties": {"title": {"type": "string"}, "body": {"type": "string"}},
     "required": ["title", "body"]}, write=True)
def _publish(title, body): return publish_note(title, body)

print("등록된 도구:", list(server.tools))

등록된 도구: ['search_news', 'publish_note']


## 3. 프로토콜 raw 호출 — 표준 메시지를 직접 보내본다
에이전트 없이, JSON-RPC 메시지를 손으로 보내 `tools/list`·`tools/call`이 어떻게 응답하는지 봅니다. 마지막은 일부러 없는 도구를 불러 **표준 오류(-32601)**를 확인합니다.

In [6]:
# ── 프로토콜 raw 호출 — 표준 메시지를 직접 보내본다 ──
# 에이전트 없이, JSON-RPC 메시지를 손으로 보내 tools/list·tools/call이 어떻게 응답하는지 봅니다. 마지막은…
def show(req):
    print("→", json.dumps(req, ensure_ascii=False))
    print("←", json.dumps(server.handle(req), ensure_ascii=False), "\n")

show({"jsonrpc": "2.0", "id": 1, "method": "tools/list"})
show({"jsonrpc": "2.0", "id": 2, "method": "tools/call",
      "params": {"name": "search_news", "arguments": {"keyword": "AI 반도체"}}})
show({"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "없는도구"}})

→ {"jsonrpc": "2.0", "id": 1, "method": "tools/list"}
← {"jsonrpc": "2.0", "id": 1, "result": {"tools": [{"name": "search_news", "description": "키워드로 뉴스 헤드라인을 검색한다", "inputSchema": {"type": "object", "properties": {"keyword": {"type": "string", "description": "검색 키워드"}}, "required": ["keyword"]}}, {"name": "publish_note", "description": "사내 게시판에 글을 발행한다(쓰기)", "inputSchema": {"type": "object", "properties": {"title": {"type": "string"}, "body": {"type": "string"}}, "required": ["title", "body"]}}]}} 

→ {"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": {"name": "search_news", "arguments": {"keyword": "AI 반도체"}}}
← {"jsonrpc": "2.0", "id": 2, "result": {"content": [{"type": "text", "text": "[\"A사 차세대 AI 가속기 공개 — 추론비용 40%↓\", \"B사 온디바이스 AI 칩 양산\"]"}]}} 

→ {"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "없는도구"}}
← {"jsonrpc": "2.0", "id": 3, "error": {"code": -32601, "message": "Unknown tool: 없는도구"}} 



## 4. Solar 에이전트 ↔ MCP 연결
이제 사람 대신 **Solar**가 도구를 고릅니다. 흐름: `tools/list`로 받은 스키마 → OpenAI tools 형식으로 변환 →
Solar가 `tool_calls` 생성 → 우리가 **`tools/call`로 실제 실행** → 결과를 다시 Solar에 넣어 최종 답변.
기본은 **읽기 도구만** 노출합니다(쓰기는 5번에서).

In [7]:
# ── Solar 에이전트 ↔ MCP 연결 ──
# 이제 사람 대신 Solar가 도구를 고릅니다. 흐름: tools/list로 받은 스키마 → OpenAI tools 형식으로 변환 …
def mcp_to_openai_tools(server, include_write=False):
    return [{"type": "function", "function": {
                "name": n, "description": t["description"], "parameters": t["inputSchema"]}}
            for n, t in server.tools.items() if include_write or not t["write"]]

def run_agent(user_msg, include_write=False, verbose=True):
    tools = mcp_to_openai_tools(server, include_write)
    messages = [{"role": "system", "content": "너는 뉴스 비서다. 필요하면 등록된 도구를 호출해 사실에 근거해 답하라."},
                {"role": "user", "content": user_msg}]
    resp = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, tool_choice="auto")
    msg = resp.choices[0].message
    if not msg.tool_calls:
        return msg.content
    messages.append(msg)
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments)
        if verbose: print(f"🛠  tools/call → {tc.function.name}({args})")
        res = server.handle({"jsonrpc": "2.0", "id": tc.id, "method": "tools/call",
                             "params": {"name": tc.function.name, "arguments": args}})
        payload = res.get("result", res.get("error"))
        messages.append({"role": "tool", "tool_call_id": tc.id,
                         "content": json.dumps(payload, ensure_ascii=False)})
    final = client.chat.completions.create(model=MODEL, messages=messages)
    return final.choices[0].message.content

print(run_agent("AI 반도체 관련 최신 뉴스 요약해줘"))

🛠  tools/call → search_news({'keyword': 'AI 반도체'})
<|tool_call:begin|>jy8e22kegh<|tool_call:name|>search_news<|tool_call:args|>{"keyword": "AI 반도체", "date": "2024-11-04", "max_results": 5}<|tool_call:end|>


## 5. 권한 스코프 — 읽기/쓰기 분리 (보안)
프로덕션에선 모든 도구를 똑같이 열어주면 안 됩니다. `publish_note`(쓰기)는 기본 비노출 →
필요할 때만, **사람 승인 게이트**를 통과해야 실행되게 합니다.

In [ ]:
# ── 권한 스코프 — 읽기/쓰기 분리 (보안) ──
# 프로덕션에선 모든 도구를 똑같이 열어주면 안 됩니다. publish_note(쓰기)는 기본 비노출 →
# (1) 기본: 쓰기 도구 비노출 → 에이전트는 발행을 할 수 없다
print("[읽기 전용]")
print(run_agent("방금 요약을 사내 게시판에 발행해줘", include_write=False), "\n")

# (2) 승인 게이트: 쓰기 호출은 사람이 승인해야 실제 실행
def guarded_call(name, args, approve: bool):
    if server.tools[name]["write"] and not approve:
        return {"error": "WRITE_BLOCKED — 사람 승인 필요"}
    return server.handle({"jsonrpc":"2.0","id":99,"method":"tools/call",
                          "params":{"name":name,"arguments":args}})

print("[승인 X]", guarded_call("publish_note", {"title":"AI 반도체 요약","body":"..."}, approve=False))
print("[승인 O]", guarded_call("publish_note", {"title":"AI 반도체 요약","body":"..."}, approve=True))

[읽기 전용]
🛠  tools/call → search_news({'keyword': '방금 요약'})
<|tool_call:begin|>o646a2g8qc<|tool_call:name|>search_news<|tool_call:args|>{"keyword": "summary published"}<|tool_call:end|> 

[승인 X] {'error': 'WRITE_BLOCKED — 사람 승인 필요'}
[승인 O] {'jsonrpc': '2.0', 'id': 99, 'result': {'content': [{'type': 'text', 'text': '{"status": "published", "title": "AI 반도체 요약", "url": "/board/9561"}'}]}}


## 정리
- MCP = **JSON-RPC 2.0** 위의 `tools/list`(스키마 광고) + `tools/call`(실행). 표준이라 **어떤 에이전트든** 우리 도구를 같은 방식으로 쓴다.
- Solar는 스키마만 보고 스스로 도구를 골라 호출했다 → **도구가 늘어도 에이전트 코드는 그대로**.
- 쓰기 도구는 **스코프 분리 + 승인 게이트**로 보호한다(9강 보안의 출발).

**NEXT:** 이렇게 만든 도구 서버들을 **여러 에이전트가 A2A로 협업**하게 묶는다.